# Scotiabank ALCO monetary-policy decision-support POC

This proof of concept (POC) evaluates how probabilistic Bank of Canada cut/hold/hike forecasts could have supported Scotiabank's Asset-Liability Committee ahead of the resolved July 15, 2026 policy decision. It combines a governed quantitative planning anchor, an optional research-grounded LLM challenge, public structural interest-rate sensitivities, and the actual BoC outcome. The POC produces decision-support information for human review; pricing, hedging, limit, and customer decisions remain within existing bank governance.


## ALCO primer

**ALCO** stands for **Asset-Liability Committee**. It is a senior bank management committee that oversees how assets, such as mortgages, business loans, and securities are funded by liabilities such as deposits and wholesale borrowing. Its purpose is to keep the balance sheet profitable and resilient while operating within liquidity, capital, market-risk, and interest-rate-risk limits.

A bank's assets and liabilities do not all respond to interest-rate changes at the same speed. A variable-rate loan may reprice quickly, while a fixed-rate mortgage may retain its rate for years. Deposit rates may also move by less than, or later than, the Bank of Canada policy rate. These timing and behavioural differences create **interest rate risk in the banking book**.

This POC focuses on two common ALCO measures:

- **Net interest income (NII):** interest earned on assets minus interest paid on funding. The 12-month NII sensitivity estimates how a rate scenario could affect near-term earnings.
- **Economic value of equity (EVE):** the present value of asset cash flows minus the present value of liability cash flows. EVE sensitivity estimates how a rate scenario could affect the longer-term economic value of the balance sheet.

NII and EVE can move in different directions because they measure different horizons. For example, higher rates might improve near-term income as assets reprice while reducing the present value of longer-duration assets. ALCO reviews both rather than relying on a single measure.

### Why a BoC probability distribution is useful

A single prediction such as "the Bank will hold" hides uncertainty. A distribution such as 20% cut, 65% hold, and 15% hike lets ALCO evaluate the base case and both tails. The notebook maps each outcome to a standardized balance-sheet scenario and probability, weights the NII and EVE effects. This helps the committee compare preparation priorities when the quantitative model and LLM challenger disagree.

The forecast is therefore an input to the ALCO process, not an automated balance-sheet decision. ALCO combines it with market pricing, liquidity and capital positions, customer behaviour, funding plans, hedge positions, limits, stress tests, and management judgment before approving an action.


## Prepare the local data cache

Run these commands from the repository root before executing the resolved experiment:

```bash
uv run python scripts/fetch_boc.py --refresh
uv run python scripts/fetch_boc_press_releases.py --year 2026
uv run python scripts/fetch_scotiabank_alco_documents.py
```

The refresh ensures the cached target-rate series contains the resolved July 15, 2026 hold and that macro and market features are available at the June 17 forecast cutoff.


To run the following Jupyter notebook via CLI, use following command 
1. `uv run jupyter lab implementations/boc_rate_decisions/05_scotiabank_alco_scenario_brief.ipynb`

## POC objective and method

1. A BoC decision distribution.
2. Scotiabank's publicly disclosed ±100 bp structural interest sensitivity.
3. Illustrative 25 bp cut/hold/hike balance-sheet scenarios.
4. A probability-weighted NII/EVE overlay and ALCO management agenda.

For a reproducible POC, the experiment adopts the assumptions in the public disclosure: an immediate sustained parallel shock, a constant balance sheet, and no mitigating management action. The disclosed ±100 bp values are scaled to standardized 25 bp decision scenarios, creating a consistent measurement layer for comparing forecast methods.


In [27]:
import importlib
import json
from datetime import datetime
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display
from aieng.forecasting.evaluation import ForecastingTask, TaskCategory
from boc_rate_decisions.analyst_agent import build_boc_research_predictor
from boc_rate_decisions.data import DIRECTION_SERIES_ID, build_boc_service
from boc_rate_decisions.predictors.logistic_baseline import BoCLogisticPredictor
import boc_rate_decisions.scotiabank_alco as scotiabank_alco_module

# Reload local POC helpers so an already-running notebook kernel sees source updates.
scotiabank_alco_module = importlib.reload(scotiabank_alco_module)
from boc_rate_decisions.scotiabank_alco import (
    SCOTIABANK_PUBLIC_SENSITIVITIES,
    SCOTIABANK_SENSITIVITY_AS_OF,
    build_alco_scenarios,
    complete_with_configured_model,
    render_alco_brief,
    scenarios_to_frame,
)


## Resolved-decision experiment configuration

The experiment forecasts the July 15, 2026 BoC announcement from a June 17, 2026 cutoff—28 days before the meeting and after Scotiabank published its Q2 2026 report on May 27. The cutoff-aware multinomial logistic predictor is the default planning anchor. Set `RUN_LLM_COMPARISON = True` to generate a research-grounded LLM challenge distribution. The actual BoC decision was **hold**, providing the outcome benchmark for forecast quality and ALCO-impact comparison.


In [28]:
def find_repo_root(start: Path | None = None) -> Path:
    candidate = (start or Path.cwd()).resolve()
    for directory in (candidate, *candidate.parents):
        if (directory / "implementations/boc_rate_decisions").is_dir():
            return directory
    raise FileNotFoundError(f"Repository root not found from {candidate}")

# Setting working directories to load cached files
REPO_ROOT = find_repo_root()
MANIFEST_PATH = REPO_ROOT / "implementations/boc_rate_decisions/scotiabank_alco_manifest.json"
SCOTIA_CACHE = REPO_ROOT / "data/reports/scotiabank_alco"
BOC_RELEASE_CACHE = REPO_ROOT / "data/reports/boc_press_releases"

# Resolved BoC decision: forecast at T-28 using only cutoff-visible information
FORECAST_AS_OF = datetime(2026, 6, 17)
MEETING_DATE = pd.Timestamp("2026-07-15")
ACTUAL_DECISION = "hold"
ACTUAL_DECISION_SOURCE = "https://www.bankofcanada.ca/2026/07/fad-press-release-2026-07-15/"

# AGENT_MODEL = "gemini-3.1-flash-lite-preview"
# AGENT_MODEL = "gemini-3.5-flash"  # Gemini advance model
AGENT_MODEL = "claude-opus-5"     # Claude advance model
RUN_LLM_COMPARISON = True
RUN_RESOLVED_LLM_INTERPRETATION = True
RUN_BACKTEST_LLM_INTERPRETATION = True
USE_MANUAL_OVERRIDE = False
MANUAL_PROBABILITIES = {"cut": 0.10, "hold": 0.70, "hike": 0.20}


## Cached public-document inventory

The manifest is committed; PDFs and provenance are cached under `data/` and are not committed.

In [29]:
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
inventory = []
for document in manifest["documents"]:
    pdf_path = SCOTIA_CACHE / f"{document['doc_id']}.pdf"
    provenance_path = SCOTIA_CACHE / "provenance" / f"{document['doc_id']}.json"
    inventory.append({
        **document,
        "cached": pdf_path.exists(),
        "cache_path": str(pdf_path),
        "provenance_cached": provenance_path.exists(),
    })
inventory_df = pd.DataFrame(inventory)
display(inventory_df[["title", "publication_date", "cached", "provenance_cached", "url"]])
if not inventory_df["cached"].all():
    display(Markdown("> Some public PDFs are not cached. Run `python3 scripts/fetch_scotiabank_alco_documents.py`; the scenario calculations still use the source-cited public sensitivity constants."))


,title,publication_date,cached,provenance_cached,url
0,Scotiabank Annual Report 2025,2025-12-02,True,True,https://www.scotiabank.com/content/dam/scotiab...
1,Scotiabank Q2 2026 Report to Shareholders,2026-05-27,True,True,https://www.scotiabank.com/content/dam/scotiab...
2,A Canadian Rates Outlook for 2026–27,2025-12-09,True,True,https://www.scotiabank.com/content/dam/scotiab...


## Public Scotiabank structural interest sensitivity

Table T27 in the Q2 2026 Report to Shareholders provides total major-currency impacts for immediate sustained ±100 bp shocks. These published values serve as the POC's controlled sensitivity layer; productionization replaces or supplements them with governed internal ALM measures at currency, product, and tenor level.


In [30]:
sensitivity_df = pd.DataFrame(SCOTIABANK_PUBLIC_SENSITIVITIES).T.reset_index(names="shock")
display(Markdown(f"**Disclosure as of:** {SCOTIABANK_SENSITIVITY_AS_OF}"))
display(sensitivity_df)


**Disclosure as of:** 2026-04-30

,shock,nii_cad_millions,eve_cad_millions
0,up_100bp,197.0,-1871.0
1,down_100bp,-189.0,1615.0


## Obtain forecasts and register the actual outcome

The quantitative distribution is produced by the cutoff-aware multinomial logistic predictor from lagged macro and market features. The optional research-grounded LLM uses the same forecast task and cutoff context as a challenger. The realized hold is represented as a one-hot outcome distribution, allowing all three views to pass through the same ALCO scenario calculation.


In [31]:
horizon_days = int((MEETING_DATE - pd.Timestamp(FORECAST_AS_OF)).days)
task = ForecastingTask(
    task_id="boc_rate_direction_scotiabank_alco",
    target_series_id=DIRECTION_SERIES_ID,
    horizons=[horizon_days],
    frequency="D",
    payload_type="categorical",
    categories=[
        TaskCategory(label="cut", value=-1),
        TaskCategory(label="hold", value=0),
        TaskCategory(label="hike", value=1),
    ],
    description="Will the Bank of Canada cut, hold, or hike at the configured meeting?",
)
service = build_boc_service(reports_dir=BOC_RELEASE_CACHE)
context = service.context(FORECAST_AS_OF)
quantitative_prediction = BoCLogisticPredictor().predict(task, context)[0]
quantitative_probabilities = quantitative_prediction.payload.probabilities
actual_probabilities = {label: float(label == ACTUAL_DECISION) for label in ("cut", "hold", "hike")}
manual_probabilities = MANUAL_PROBABILITIES.copy()
anchor_source = "manual_override" if USE_MANUAL_OVERRIDE else "quantitative_logistic"
anchor_probabilities = manual_probabilities if USE_MANUAL_OVERRIDE else quantitative_probabilities
llm_probabilities = None
llm_rationale = "LLM comparison not run."

if RUN_LLM_COMPARISON:
    prediction = build_boc_research_predictor(model=AGENT_MODEL).predict(task, context)[0]
    llm_probabilities = prediction.payload.probabilities
    llm_rationale = prediction.metadata.get("rationale", "No rationale returned.")

distribution_rows = [
    {"source": "quantitative_logistic", **quantitative_probabilities},
    {"source": "actual_boc_outcome", **actual_probabilities},
]
if USE_MANUAL_OVERRIDE:
    distribution_rows.append({"source": "manual_override", **manual_probabilities})
if llm_probabilities is not None:
    distribution_rows.append({"source": f"llm:{AGENT_MODEL}", **llm_probabilities})
distribution_df = pd.DataFrame(distribution_rows).set_index("source")
display(distribution_df.style.format("{:.1%}"))
display(Markdown(
    f"**Active planning anchor:** `{anchor_source}`  \n"
    f"**Actual BoC decision:** `{ACTUAL_DECISION}` ([official announcement]({ACTUAL_DECISION_SOURCE}))  \n"
    f"**LLM rationale:** {llm_rationale}"
))


,cut,hold,hike
source,,,
quantitative_logistic,2.1%,83.9%,14.0%
actual_boc_outcome,0.0%,100.0%,0.0%
llm:claude-opus-5,5.0%,88.0%,7.0%


**Active planning anchor:** `quantitative_logistic`  
**Actual BoC decision:** `hold` ([official announcement](https://www.bankofcanada.ca/2026/07/fad-press-release-2026-07-15/))  
**LLM rationale:** Setup: the policy rate has been at 2.25% since the October 2025 cut, with five consecutive holds (2025-12-10, 2026-01-28, 2026-03-18, 2026-04-29, 2026-06-10). Rate momentum is 0.0 — the easing cycle that ran from June 2024 through October 2025 has clearly been paused, and the Bank has settled into a wait-and-see stance amid two overlapping shocks (US tariffs/trade uncertainty and the Middle East/Iran war energy shock).

Documentary evidence (the three supplied press releases, 2026-03-18_en, 2026-04-29_en, 2026-06-10_en) is unusually consistent and points to continuation of the hold. In 2026-06-10_en the Bank states it 'decided to maintain the policy rate at 2.25%', notes Q1 GDP edged down 0.1%, employment 'little changed since the start of the year', unemployment fluctuating in the 6½%-7% range (6.6% in May), and that 'the economy is expected to remain in excess supply.' At the same time CPI inflation reached 2.8% in April and total inflation is 'expected to hover around 3% in the near term'. Governing Council explicitly says it is 'continuing to look through the war's near-term impact on headline inflation, but will not let higher energy prices become persistent inflation' — a deliberately two-sided, neutral formulation with no easing or tightening bias. The stock phrase 'As the outlook evolves, we stand ready to respond as needed' has been repeated verbatim across all three statements, which historically signals no pre-commitment to a move.

The two tails are asymmetrically constrained. A cut is blocked by headline inflation near 3% and rising near-term inflation expectations (2026-04-29_en), plus already-loosened financial conditions and a weaker Canadian dollar (2026-06-10_en) — cutting into a 3% headline print would risk the price-stability credibility the Bank keeps invoking. A hike is blocked by an economy in excess supply, contracting/flat GDP, a soft labour market, and core inflation 'around 2%' with the share of CPI components above 3% near its historical average — i.e., no evidence yet of the broad pass-through that would trigger tightening. The macro snapshot's +0.5 yield spread (2-year GoC roughly 0.5pp above the 2.25% policy rate) is the one signal that argues the market has priced out cuts and put some probability on eventual tightening, and unemployment momentum of -0.4 (jobless rate drifting down within the 6½-7% band) marginally supports that direction. That justifies putting the residual tail weight modestly more on hike than cut, but the July 15 meeting comes with an MPR, and the Bank's institutional gradualism means it would almost certainly pre-signal a reversal from cuts to hikes before delivering one — no such signal appears in the June statement. Direct cut-to-hike reversals essentially never occur without a communicated pivot.

Combining the ~76% unconditional hold base rate with a five-meeting hold streak, an explicitly neutral June statement, excess supply, and energy-driven headline inflation the Bank says it is looking through, I put hold near 0.88, with a slightly fatter hike tail (0.07) than cut tail (0.05) reflecting the positive 2-year spread and the Bank's warning about not letting energy prices become persistent inflation.

## Compare forecast distributions with the actual decision

The table compares the quantitative anchor and LLM challenger. The actual decision is shown in the distribution table above as 100% hold. Total variation distance summarizes anchor-versus-LLM disagreement from 0 (identical) to 1 (no overlapping probability mass), while each method's probability assigned to hold shows how strongly it supported the realized outcome.


In [32]:
if llm_probabilities is not None:
    probability_comparison = pd.DataFrame({
        "decision": ["cut", "hold", "hike"],
        "anchor": [anchor_probabilities[key] for key in ("cut", "hold", "hike")],
        "llm": [llm_probabilities[key] for key in ("cut", "hold", "hike")],
    })
    probability_comparison["llm_minus_anchor"] = probability_comparison["llm"] - probability_comparison["anchor"]
    total_variation_distance = 0.5 * probability_comparison["llm_minus_anchor"].abs().sum()
    display(probability_comparison.style.format({
        "anchor": "{:.1%}", "llm": "{:.1%}", "llm_minus_anchor": "{:+.1%}",
    }))
    display(Markdown(f"**Anchor:** `{anchor_source}`  \n**Total variation distance:** `{total_variation_distance:.1%}`"))
else:
    display(Markdown("> Set `RUN_LLM_COMPARISON = True` to populate the LLM-minus-anchor comparison."))

outcome_evaluation_rows = []
forecast_distributions = {"quantitative_logistic": quantitative_probabilities}
if llm_probabilities is not None:
    forecast_distributions[f"llm:{AGENT_MODEL}"] = llm_probabilities
for method, probabilities in forecast_distributions.items():
    predicted_decision = max(probabilities, key=probabilities.get)
    outcome_evaluation_rows.append({
        "method": method,
        "predicted_decision": predicted_decision,
        "actual_decision": ACTUAL_DECISION,
        "correct": predicted_decision == ACTUAL_DECISION,
        "probability_assigned_to_actual": probabilities[ACTUAL_DECISION],
    })
outcome_evaluation = pd.DataFrame(outcome_evaluation_rows)
display(outcome_evaluation.style.format({"probability_assigned_to_actual": "{:.1%}"}))


,decision,anchor,llm,llm_minus_anchor
0,cut,2.1%,5.0%,+2.9%
1,hold,83.9%,88.0%,+4.1%
2,hike,14.0%,7.0%,-7.0%


**Anchor:** `quantitative_logistic`  
**Total variation distance:** `7.0%`

,method,predicted_decision,actual_decision,correct,probability_assigned_to_actual
0,quantitative_logistic,hold,hold,True,83.9%
1,llm:claude-opus-5,hold,hold,True,88.0%


## Translate the distributions into ALCO scenarios


In [33]:
scenario_sets = {
    "quantitative_logistic": build_alco_scenarios(quantitative_probabilities, move_bp=25),
    "actual_boc_outcome": build_alco_scenarios(actual_probabilities, move_bp=25),
}
if USE_MANUAL_OVERRIDE:
    scenario_sets["manual_override"] = build_alco_scenarios(manual_probabilities, move_bp=25)
if llm_probabilities is not None:
    scenario_sets[f"llm:{AGENT_MODEL}"] = build_alco_scenarios(llm_probabilities, move_bp=25)

scenario_frames = []
for source, source_scenarios in scenario_sets.items():
    frame = scenarios_to_frame(source_scenarios)
    frame.insert(0, "source", source)
    scenario_frames.append(frame)
scenario_df = pd.concat(scenario_frames, ignore_index=True)
display(scenario_df.style.format({
    "probability": "{:.1%}",
    "nii_12m_impact_cad_millions": "{:+.1f}",
    "eve_impact_cad_millions": "{:+.1f}",
    "weighted_nii_cad_millions": "{:+.1f}",
    "weighted_eve_cad_millions": "{:+.1f}",
}))


,source,decision,probability,policy_move_bp,nii_12m_impact_cad_millions,eve_impact_cad_millions,weighted_nii_cad_millions,weighted_eve_cad_millions,management_focus
0,quantitative_logistic,cut,2.1%,-25,-47.2,+403.8,-1.0,+8.5,"Review deposit floors, term-deposit migration, asset repricing, and prepayment assumptions."
1,quantitative_logistic,hold,83.9%,0,+0.0,+0.0,+0.0,+0.0,"Maintain base plan; monitor funding mix, customer beta, and incoming inflation/labour data."
2,quantitative_logistic,hike,14.0%,25,+49.2,-467.8,+6.9,-65.3,"Review duration exposure, borrower affordability, variable-rate credit, and hedge readiness."
3,actual_boc_outcome,cut,0.0%,-25,-47.2,+403.8,-0.0,+0.0,"Review deposit floors, term-deposit migration, asset repricing, and prepayment assumptions."
4,actual_boc_outcome,hold,100.0%,0,+0.0,+0.0,+0.0,+0.0,"Maintain base plan; monitor funding mix, customer beta, and incoming inflation/labour data."
5,actual_boc_outcome,hike,0.0%,25,+49.2,-467.8,+0.0,-0.0,"Review duration exposure, borrower affordability, variable-rate credit, and hedge readiness."
6,llm:claude-opus-5,cut,5.0%,-25,-47.2,+403.8,-2.4,+20.2,"Review deposit floors, term-deposit migration, asset repricing, and prepayment assumptions."
7,llm:claude-opus-5,hold,88.0%,0,+0.0,+0.0,+0.0,+0.0,"Maintain base plan; monitor funding mix, customer beta, and incoming inflation/labour data."
8,llm:claude-opus-5,hike,7.0%,25,+49.2,-467.8,+3.4,-32.7,"Review duration exposure, borrower affordability, variable-rate credit, and hedge readiness."


## Compare probability-weighted balance-sheet overlays

This stage holds the POC balance-sheet sensitivity assumptions constant and compares the ex-ante, probability-weighted NII/EVE overlays with the standardized overlay for the realized BoC hold. Deltas versus the actual outcome show how each forecast method would have changed ALCO's pre-meeting planning view.


In [34]:
impact_comparison = (
    scenario_df.groupby("source", as_index=False)
    .agg(
        expected_nii_cad_millions=("weighted_nii_cad_millions", "sum"),
        expected_eve_cad_millions=("weighted_eve_cad_millions", "sum"),
    )
)
anchor_row = impact_comparison.loc[impact_comparison["source"] == anchor_source].iloc[0]
actual_row = impact_comparison.loc[impact_comparison["source"] == "actual_boc_outcome"].iloc[0]
impact_comparison["nii_delta_vs_anchor"] = impact_comparison["expected_nii_cad_millions"] - anchor_row["expected_nii_cad_millions"]
impact_comparison["eve_delta_vs_anchor"] = impact_comparison["expected_eve_cad_millions"] - anchor_row["expected_eve_cad_millions"]
impact_comparison["nii_error_vs_actual"] = impact_comparison["expected_nii_cad_millions"] - actual_row["expected_nii_cad_millions"]
impact_comparison["eve_error_vs_actual"] = impact_comparison["expected_eve_cad_millions"] - actual_row["expected_eve_cad_millions"]
display(impact_comparison.style.format({column: "{:+.1f}" for column in impact_comparison.columns if column != "source"}))


,source,expected_nii_cad_millions,expected_eve_cad_millions,nii_delta_vs_anchor,eve_delta_vs_anchor,nii_error_vs_actual,eve_error_vs_actual
0,actual_boc_outcome,+0.0,+0.0,-5.9,+56.8,+0.0,+0.0
1,llm:claude-opus-5,+1.1,-12.6,-4.8,+44.3,+1.1,-12.6
2,quantitative_logistic,+5.9,-56.8,+0.0,+0.0,+5.9,-56.8


## Plain-language LLM interpretation of the resolved experiment

This is a separate interpretation call after the numerical calculations are complete. It uses the model selected by `AGENT_MODEL` and receives the forecast comparison, the actual hold, and the standardized NII/EVE overlays. It does not recalculate results or change the planning anchor; its role is to explain the evidence to a non-banking audience.


In [35]:
def call_plain_language_interpreter(*, title: str, evidence: str) -> str:
    return complete_with_configured_model(
        model=AGENT_MODEL,
        max_tokens=1200,
        messages=[
            {
                "role": "system",
                "content": (
                    "You explain a bank ALCO proof-of-concept to readers with no banking background. "
                    "Use only the supplied evidence. Explain cut/hold/hike probabilities, what the models "
                    "predicted, what actually happened, and why forecast uncertainty changes the NII and EVE "
                    "planning overlays. Define NII and EVE briefly. Clearly distinguish standardized scenario "
                    "impacts from Scotiabank's realized financial results. Do not recommend trades, pricing, "
                    "hedges, or balance-sheet actions. Use concise Markdown with: Summary, Model comparison, "
                    "ALCO meaning, and Takeaway. Mention exact numbers when useful."
                ),
            },
            {"role": "user", "content": f"# {title}\n\n{evidence}"},
        ],
    )

if RUN_RESOLVED_LLM_INTERPRETATION:
    resolved_evidence = (
        f"Forecast cutoff: {FORECAST_AS_OF.date()}\n"
        f"BoC meeting: {MEETING_DATE.date()}\n"
        f"Actual decision: {ACTUAL_DECISION}\n\n"
        "Forecast outcome evaluation:\n" + outcome_evaluation.to_json(orient="records", indent=2)
        + "\n\nProbability distributions:\n" + distribution_df.reset_index().to_json(orient="records", indent=2)
        + "\n\nALCO overlay comparison, CAD millions:\n" + impact_comparison.to_json(orient="records", indent=2)
    )
    resolved_interpretation = call_plain_language_interpreter(
        title="Resolved July 15, 2026 BoC decision", evidence=resolved_evidence
    )
    display(Markdown(resolved_interpretation))
else:
    display(Markdown("> Set `RUN_RESOLVED_LLM_INTERPRETATION = True` to generate this explanation."))


# BoC July 15, 2026 Decision — ALCO Proof-of-Concept Readout

## Summary

- **Setup:** Two forecasting methods were run as of a **2026-06-17 cutoff** to predict the Bank of Canada's **2026-07-15** policy rate decision across three outcomes: **cut**, **hold**, or **hike**.
- **Actual decision: hold.** In outcome terms the realized distribution is cut 0.0 / hold 1.0 / hike 0.0.
- **Both models called it correctly:**
  - `quantitative_logistic`: cut **2.11%**, hold **83.93%**, hike **13.96%**
  - `llm:claude-opus-5`: cut **5%**, hold **88%**, hike **7%**
- Both put the bulk of probability on hold; they differed mainly in how much weight they left on a **hike** (13.96% vs 7%). That residual uncertainty — not the headline call — is what moved the planning overlays.

## Model comparison

| Method | Predicted | Correct | P(actual outcome) | Cut | Hold | Hike |
|---|---|---|---|---|---|---|
| quantitative_log

## Render the human-review ALCO brief


In [36]:
anchor_brief = render_alco_brief(
    scenario_sets[anchor_source],
    meeting_date=MEETING_DATE.date().isoformat(),
    forecast_as_of=FORECAST_AS_OF.date().isoformat(),
    evidence_titles=[item["title"] for item in inventory],
)
display(Markdown(f"## Active planning anchor ({anchor_source})\n\n" + anchor_brief))
if llm_probabilities is not None:
    llm_brief = render_alco_brief(
        scenario_sets[f"llm:{AGENT_MODEL}"],
        meeting_date=MEETING_DATE.date().isoformat(),
        forecast_as_of=FORECAST_AS_OF.date().isoformat(),
        evidence_titles=[item["title"] for item in inventory],
    )
    display(Markdown(f"## LLM challenge view ({AGENT_MODEL})\n\n" + llm_brief))
actual_brief = render_alco_brief(
    scenario_sets["actual_boc_outcome"],
    meeting_date=MEETING_DATE.date().isoformat(),
    forecast_as_of=FORECAST_AS_OF.date().isoformat(),
    evidence_titles=["Bank of Canada July 15, 2026 interest-rate announcement"],
)
display(Markdown("## Actual-outcome ALCO benchmark\n\n" + actual_brief))


## Active planning anchor (quantitative_logistic)

# Illustrative Scotiabank ALCO Monetary-Policy Scenario Brief

**Forecast as of:** 2026-06-17  
**BoC meeting:** 2026-07-15  
**Highest-probability decision:** HOLD (83.9%)

## Scenario distribution and disclosed-sensitivity overlay

| Decision | Probability | Policy move (bp) | 12m NII impact (CAD mm) | EVE impact (CAD mm) |
|---|---:|---:|---:|---:|
| Cut | 2.1% | -25 | -47.2 | +403.8 |
| Hold | 83.9% | +0 | +0.0 | +0.0 |
| Hike | 14.0% | +25 | +49.2 | -467.8 |

**Probability-weighted illustrative impact:** NII +5.9 CAD mm; EVE -56.8 CAD mm.

## Management focus by scenario

- **Cut:** Review deposit floors, term-deposit migration, asset repricing, and prepayment assumptions.
- **Hold:** Maintain base plan; monitor funding mix, customer beta, and incoming inflation/labour data.
- **Hike:** Review duration exposure, borrower affordability, variable-rate credit, and hedge readiness.

## Cached public evidence

- Scotiabank Annual Report 2025
- Scotiabank Q2 2026 Report to Shareholders
- A Canadian Rates Outlook for 2026–27

## Required interpretation controls

- Public-data prototype only; not Scotiabank internal ALCO analysis or advice.
- Sensitivities come from the Q2 2026 public disclosure and are linearly
  scaled from an immediate, sustained ±100 bp parallel shock.
- The disclosure assumes a constant balance sheet and no mitigating management
  action; an actual 25 bp policy move is not equivalent to a parallel curve shock.
- Probability weighting is a decision-support summary, not an instruction to
  price products, alter hedges, or take market positions.
- ALCO, Treasury, Finance, and independent risk/model validation must review assumptions and overlays.


## LLM challenge view (claude-opus-5)

# Illustrative Scotiabank ALCO Monetary-Policy Scenario Brief

**Forecast as of:** 2026-06-17  
**BoC meeting:** 2026-07-15  
**Highest-probability decision:** HOLD (88.0%)

## Scenario distribution and disclosed-sensitivity overlay

| Decision | Probability | Policy move (bp) | 12m NII impact (CAD mm) | EVE impact (CAD mm) |
|---|---:|---:|---:|---:|
| Cut | 5.0% | -25 | -47.2 | +403.8 |
| Hold | 88.0% | +0 | +0.0 | +0.0 |
| Hike | 7.0% | +25 | +49.2 | -467.8 |

**Probability-weighted illustrative impact:** NII +1.1 CAD mm; EVE -12.6 CAD mm.

## Management focus by scenario

- **Cut:** Review deposit floors, term-deposit migration, asset repricing, and prepayment assumptions.
- **Hold:** Maintain base plan; monitor funding mix, customer beta, and incoming inflation/labour data.
- **Hike:** Review duration exposure, borrower affordability, variable-rate credit, and hedge readiness.

## Cached public evidence

- Scotiabank Annual Report 2025
- Scotiabank Q2 2026 Report to Shareholders
- A Canadian Rates Outlook for 2026–27

## Required interpretation controls

- Public-data prototype only; not Scotiabank internal ALCO analysis or advice.
- Sensitivities come from the Q2 2026 public disclosure and are linearly
  scaled from an immediate, sustained ±100 bp parallel shock.
- The disclosure assumes a constant balance sheet and no mitigating management
  action; an actual 25 bp policy move is not equivalent to a parallel curve shock.
- Probability weighting is a decision-support summary, not an instruction to
  price products, alter hedges, or take market positions.
- ALCO, Treasury, Finance, and independent risk/model validation must review assumptions and overlays.


## Actual-outcome ALCO benchmark

# Illustrative Scotiabank ALCO Monetary-Policy Scenario Brief

**Forecast as of:** 2026-06-17  
**BoC meeting:** 2026-07-15  
**Highest-probability decision:** HOLD (100.0%)

## Scenario distribution and disclosed-sensitivity overlay

| Decision | Probability | Policy move (bp) | 12m NII impact (CAD mm) | EVE impact (CAD mm) |
|---|---:|---:|---:|---:|
| Cut | 0.0% | -25 | -47.2 | +403.8 |
| Hold | 100.0% | +0 | +0.0 | +0.0 |
| Hike | 0.0% | +25 | +49.2 | -467.8 |

**Probability-weighted illustrative impact:** NII +0.0 CAD mm; EVE +0.0 CAD mm.

## Management focus by scenario

- **Cut:** Review deposit floors, term-deposit migration, asset repricing, and prepayment assumptions.
- **Hold:** Maintain base plan; monitor funding mix, customer beta, and incoming inflation/labour data.
- **Hike:** Review duration exposure, borrower affordability, variable-rate credit, and hedge readiness.

## Cached public evidence

- Bank of Canada July 15, 2026 interest-rate announcement

## Required interpretation controls

- Public-data prototype only; not Scotiabank internal ALCO analysis or advice.
- Sensitivities come from the Q2 2026 public disclosure and are linearly
  scaled from an immediate, sustained ±100 bp parallel shock.
- The disclosure assumes a constant balance sheet and no mitigating management
  action; an actual 25 bp policy move is not equivalent to a parallel curve shock.
- Probability weighting is a decision-support summary, not an instruction to
  price products, alter hedges, or take market positions.
- ALCO, Treasury, Finance, and independent risk/model validation must review assumptions and overlays.


## Productionization roadmap

The POC establishes the forecast-to-ALCO workflow, comparison metrics, provenance controls, and human-review output. Productionization proceeds through the following workstreams:

1. **Data and ALM integration:** connect governed internal currency, product, tenor, deposit-beta, prepayment, optionality, hedge, and funding data.
2. **Scenario enrichment:** add non-parallel yield-curve shocks, market-implied CORRA/OIS paths, multiple move sizes, and Treasury/Finance base plans and management actions.
3. **Risk quantification:** replace point scaling with approved NII/EVE engines, confidence ranges, limit utilization, and stress aggregation.
4. **Model governance:** complete independent validation, model inventory registration, challenger thresholds, override reason codes, change control, and ALCO approval.
5. **Operating model:** schedule data refreshes and forecast runs, retain immutable inputs and outputs, register forecasts prospectively, and monitor calibration, RPS, stability, and post-meeting ALCO impact errors.
6. **Controlled deployment:** begin in shadow mode, advance through parallel run and user acceptance, and permit broader use only after governance and performance gates are met.


---
# POC historical forecast-to-ALCO experiment

This experiment extends the resolved July 2026 case above across additional historical decisions. For each meeting in the historical smoke specification, it reconstructs the information available 28 days before the decision, obtains logistic-regression probabilities, optionally obtains research-grounded LLM probabilities, and compares both with the realized BoC action. It then translates each forecast distribution and the realized action into the same standardized Scotiabank NII/EVE overlay.

**POC experiment design:** the outcome benchmark is a one-hot cut/hold/hike distribution, with cut and hike standardized to 25 bp. The ALCO outcome measure applies the same public parallel-shock sensitivity to every method and to the realized direction, ensuring an apples-to-apples comparison. Model assessment combines historical workflow validation with prospectively registered forecasts during the productionization phase.


In [37]:
import yaml
from aieng.forecasting.evaluation import BacktestSpec, backtest

HISTORICAL_SPEC_PATH = REPO_ROOT / "implementations/boc_rate_decisions/specs/boc_rate_direction_smoke.yaml"
RUN_HISTORICAL_LLM_BACKTEST = True
HISTORICAL_MOVE_BP = 25

with HISTORICAL_SPEC_PATH.open(encoding="utf-8") as file:
    historical_spec = BacktestSpec.model_validate(yaml.safe_load(file))

display(Markdown(
    f"**Historical origins:** `{len(historical_spec.origin_dates)}`  \
"
    f"**Forecast lead:** `{historical_spec.task.horizons[0]} days`  \
"
    f"**Historical LLM calls enabled:** `{RUN_HISTORICAL_LLM_BACKTEST}`"
))


**Historical origins:** `3`  **Forecast lead:** `28 days`  **Historical LLM calls enabled:** `True`

## Run cutoff-aware historical predictions

Logistic regression always runs locally. Set `RUN_HISTORICAL_LLM_BACKTEST = True` to add one research-grounded model call per historical origin. Each prediction receives a context whose observations and cached documents are restricted to its forecast cutoff.


In [38]:
historical_predictors = {"logistic_regression": BoCLogisticPredictor()}
if RUN_HISTORICAL_LLM_BACKTEST:
    historical_predictors[f"research_llm:{AGENT_MODEL}"] = build_boc_research_predictor(model=AGENT_MODEL)

historical_results = {}
for method_name, method in historical_predictors.items():
    print(f"Running {method_name}...")
    historical_results[method_name] = backtest(method, historical_spec, service)
print("Historical prediction backtest complete.")


Running logistic_regression...
Running research_llm:claude-opus-5...
Historical prediction backtest complete.


## Predicted probabilities versus actual decisions

The realized decision is shown beside each method's full probability distribution and highest-probability decision. RPS evaluates the ordered probability distribution; lower is better.


In [39]:
resolved_decisions = service.get_series(historical_spec.task.target_series_id, as_of=datetime.now())
value_to_label = {category.value: category.label for category in historical_spec.task.categories}
actual_by_meeting = {
    pd.Timestamp(timestamp).date(): value_to_label[float(value)]
    for timestamp, value in zip(
        resolved_decisions["timestamp"], resolved_decisions["value"], strict=True
    )
}

forecast_rows = []
for method_name, result in historical_results.items():
    for prediction, rps in zip(result.predictions, result.scores, strict=True):
        meeting = pd.Timestamp(prediction.forecast_date).date()
        probabilities = prediction.payload.probabilities
        predicted_decision = max(probabilities, key=probabilities.get)
        actual_decision = actual_by_meeting[meeting]
        forecast_rows.append({
            "meeting": meeting,
            "forecast_origin": pd.Timestamp(prediction.as_of).date(),
            "method": method_name,
            "predicted_decision": predicted_decision,
            "actual_decision": actual_decision,
            "correct": predicted_decision == actual_decision,
            "p_cut": probabilities["cut"],
            "p_hold": probabilities["hold"],
            "p_hike": probabilities["hike"],
            "rps": rps,
        })

historical_forecasts = pd.DataFrame(forecast_rows).sort_values(["meeting", "method"])
display(historical_forecasts.style.format({
    "p_cut": "{:.1%}", "p_hold": "{:.1%}", "p_hike": "{:.1%}", "rps": "{:.3f}",
}))


,meeting,forecast_origin,method,predicted_decision,actual_decision,correct,p_cut,p_hold,p_hike,rps
0,2024-04-10,2024-03-13,logistic_regression,hold,hold,True,4.2%,94.8%,1.0%,0.002
3,2024-04-10,2024-03-13,research_llm:claude-opus-5,hold,hold,True,6.0%,93.5%,0.5%,0.004
1,2024-06-05,2024-05-08,logistic_regression,hold,cut,False,3.5%,95.4%,1.1%,0.931
4,2024-06-05,2024-05-08,research_llm:claude-opus-5,cut,cut,True,52.0%,47.0%,1.0%,0.230
2,2024-09-04,2024-08-07,logistic_regression,cut,cut,True,58.6%,41.3%,0.1%,0.171
5,2024-09-04,2024-08-07,research_llm:claude-opus-5,cut,cut,True,89.0%,10.5%,0.5%,0.012


## Forecast-implied versus realized ALCO overlays

For a forecast, the overlay is probability-weighted across cut/hold/hike. For the outcome benchmark, the realized category receives 100% probability. `NII/EVE error vs actual` measures how far each method's ex-ante planning overlay was from the standardized realized-direction overlay. In production, the same evaluation contract can be connected to approved internal ALM results and realized management actions.


In [40]:
def expected_alco_impacts(probabilities: dict[str, float]) -> tuple[float, float]:
    scenarios = build_alco_scenarios(probabilities, move_bp=HISTORICAL_MOVE_BP)
    frame = scenarios_to_frame(scenarios)
    expected_nii = frame["weighted_nii_cad_millions"].sum()
    expected_eve = frame["weighted_eve_cad_millions"].sum()
    return expected_nii, expected_eve

alco_rows = []
for row in historical_forecasts.to_dict(orient="records"):
    forecast_distribution = {
        "cut": row["p_cut"], "hold": row["p_hold"], "hike": row["p_hike"]
    }
    actual_distribution = {label: float(label == row["actual_decision"]) for label in ("cut", "hold", "hike")}
    forecast_nii, forecast_eve = expected_alco_impacts(forecast_distribution)
    actual_nii, actual_eve = expected_alco_impacts(actual_distribution)
    alco_rows.append({
        "meeting": row["meeting"],
        "method": row["method"],
        "predicted_decision": row["predicted_decision"],
        "actual_decision": row["actual_decision"],
        "forecast_expected_nii_cad_m": forecast_nii,
        "actual_scenario_nii_cad_m": actual_nii,
        "nii_error_vs_actual_cad_m": forecast_nii - actual_nii,
        "forecast_expected_eve_cad_m": forecast_eve,
        "actual_scenario_eve_cad_m": actual_eve,
        "eve_error_vs_actual_cad_m": forecast_eve - actual_eve,
    })

historical_alco = pd.DataFrame(alco_rows).sort_values(["meeting", "method"])
display(historical_alco.style.format({
    column: "{:+.1f}" for column in historical_alco.columns if column.endswith("_cad_m")
}))

historical_summary = (
    historical_alco.assign(
        abs_nii_error=lambda frame: frame["nii_error_vs_actual_cad_m"].abs(),
        abs_eve_error=lambda frame: frame["eve_error_vs_actual_cad_m"].abs(),
    )
    .groupby("method", as_index=False)
    .agg(
        meetings=("meeting", "size"),
        mean_absolute_nii_error_cad_m=("abs_nii_error", "mean"),
        mean_absolute_eve_error_cad_m=("abs_eve_error", "mean"),
    )
)
forecast_quality = historical_forecasts.groupby("method", as_index=False).agg(
    point_accuracy=("correct", "mean"), mean_rps=("rps", "mean")
)
historical_summary = historical_summary.merge(forecast_quality, on="method")
display(historical_summary.style.format({
    "point_accuracy": "{:.1%}", "mean_rps": "{:.3f}",
    "mean_absolute_nii_error_cad_m": "{:.1f}",
    "mean_absolute_eve_error_cad_m": "{:.1f}",
}))


,meeting,method,predicted_decision,actual_decision,forecast_expected_nii_cad_m,actual_scenario_nii_cad_m,nii_error_vs_actual_cad_m,forecast_expected_eve_cad_m,actual_scenario_eve_cad_m,eve_error_vs_actual_cad_m
0,2024-04-10,logistic_regression,hold,hold,-1.5,+0.0,-1.5,+12.2,+0.0,+12.2
1,2024-04-10,research_llm:claude-opus-5,hold,hold,-2.6,+0.0,-2.6,+21.9,+0.0,+21.9
2,2024-06-05,logistic_regression,hold,cut,-1.1,-47.2,+46.1,+9.1,+403.8,-394.7
3,2024-06-05,research_llm:claude-opus-5,cut,cut,-24.1,-47.2,+23.2,+205.3,+403.8,-198.5
4,2024-09-04,logistic_regression,cut,cut,-27.7,-47.2,+19.6,+236.4,+403.8,-167.3
5,2024-09-04,research_llm:claude-opus-5,cut,cut,-41.8,-47.2,+5.4,+357.0,+403.8,-46.8


,method,meetings,mean_absolute_nii_error_cad_m,mean_absolute_eve_error_cad_m,point_accuracy,mean_rps
0,logistic_regression,3,22.4,191.4,66.7%,0.368
1,research_llm:claude-opus-5,3,10.4,89.0,100.0%,0.082


## Plain-language LLM interpretation of the backtest

This second interpretation call compares logistic regression with the research-grounded LLM across all backtest meetings. It explains accuracy, RPS, probability differences, and ALCO overlay errors in accessible language while keeping the calculated tables as the source of truth.


In [41]:
has_historical_llm = historical_forecasts["method"].str.startswith("research_llm:").any()
if RUN_BACKTEST_LLM_INTERPRETATION and has_historical_llm:
    backtest_evidence = (
        "Meeting-level forecasts and actual outcomes:\n"
        + historical_forecasts.to_json(orient="records", indent=2, date_format="iso")
        + "\n\nMeeting-level ALCO overlays, CAD millions:\n"
        + historical_alco.to_json(orient="records", indent=2, date_format="iso")
        + "\n\nMethod summary:\n"
        + historical_summary.to_json(orient="records", indent=2)
    )
    backtest_interpretation = call_plain_language_interpreter(
        title="Historical logistic-regression versus research-LLM comparison",
        evidence=backtest_evidence,
    )
    display(Markdown(backtest_interpretation))
elif RUN_BACKTEST_LLM_INTERPRETATION:
    display(Markdown(
        "> Enable `RUN_HISTORICAL_LLM_BACKTEST` and rerun the backtest before requesting "
        "a logistic-versus-LLM interpretation."
    ))
else:
    display(Markdown("> Set `RUN_BACKTEST_LLM_INTERPRETATION = True` to generate this explanation."))


No interpretation returned.

## POC success criteria and transition decision

The POC evaluates both forecasting quality and balance-sheet decision relevance. A candidate method advances when it demonstrates lower RPS, stable calibration, and smaller NII/EVE overlay errors across unseen meetings while producing reviewable evidence and rationale. Point accuracy remains a supporting measure because two methods can select the same decision while assigning materially different tail probabilities and preparation ranges.

The next experiment expands deterministic testing to the full historical specification and runs logistic and LLM forecasts prospectively in shadow mode. Promotion to a production pilot would be based on pre-agreed thresholds for data completeness, probability calibration, challenger disagreement, ALCO impact error, operational reliability, reviewer acceptance, and model-risk approval.
